### Ejercicio 1 — Comparar inferSchema vs schema explícito (medición y observación)
Lee sales_simple.csv de dos formas:

(A) usando inferSchema=True y (B) usando un schema explícito (StructType). Mide el tiempo de lectura en cada caso y observa printSchema() y count(). Piensa cuál usarías en desarrollo y cuál en producción.

Objetivo
Entender la diferencia práctica entre inferSchema (conveniente pero hace una pasada extra) y schema explícito (reproducible y más eficiente en ETLs grandes).

In [24]:
# importamos librerias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F # para seleccionar filas
from pyspark.sql.types import StructType, IntegerType, StringType, StructField, DoubleType, DateType
import time

In [25]:
# creamos sesión spark
spark = (SparkSession.builder
         .appName("aplicación_1") # nombre de la aplicación
         .master("local[*]") # seleccionamos como ejecución la cpu local
         .config("spark.driver.memory", "2g") #memoria del driver program
         .getOrCreate()
)

print(f"versión spark: {spark.version}")

versión spark: 4.2.0


In [26]:
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")

In [27]:
# leemos el dataset -> forma 1
tiempo1 = time.time()
df_forma1 = spark.read.csv("sales_simple.csv", header=True, inferSchema=True, sep=",")
df_forma1.show(3)

+---+----------+--------+------+--------+
| id|      date| product| price|quantity|
+---+----------+--------+------+--------+
|  1|2024-01-10|Portátil|1200.0|       1|
|  2|2024-01-11|   Ratón|  25.0|       2|
|  3|2024-01-12| Teclado|  75.0|       1|
+---+----------+--------+------+--------+
only showing top 3 rows


In [28]:
# contamos numero de filas
df_forma1.count()
tiempo2 = time.time() #medimos tiempo

In [29]:
tiempo_resultado = tiempo2 - tiempo1
print(tiempo_resultado)

0.2077019214630127


In [30]:
df_forma1.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)



In [31]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
])

tiempo1 = time.time()
df_forma2 = spark.read.csv("sales_simple.csv", header=True, schema=schema, sep=",")
df_forma2.show(3)

+---+----------+--------+------+--------+
| id|      date| product| price|quantity|
+---+----------+--------+------+--------+
|  1|2024-01-10|Portátil|1200.0|       1|
|  2|2024-01-11|   Ratón|  25.0|       2|
|  3|2024-01-12| Teclado|  75.0|       1|
+---+----------+--------+------+--------+
only showing top 3 rows


In [32]:
df_forma2.count()
tiempo2 = time.time()

In [33]:
tiempo_resultado = tiempo2 - tiempo1
print(tiempo_resultado)

0.11084389686584473


In [34]:
df_forma2.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)



### Ejercicio 2 — Añadir weekday y agrupar: ventas totales por día de la semana
Crea la columna total = price * quantity. Añade una columna weekday (día de la semana) a partir de date y agrupa para obtener la suma de total por weekday. Ordena el resultado por el total descendente.

Objetivo
Practicar withColumn, to_date/date_format y groupBy + agregaciones.

In [35]:
# creamos columna total
df = df_forma1.withColumn("total", F.col("price") * F.col("quantity"))
df.show(5)

+---+----------+--------+------+--------+------+
| id|      date| product| price|quantity| total|
+---+----------+--------+------+--------+------+
|  1|2024-01-10|Portátil|1200.0|       1|1200.0|
|  2|2024-01-11|   Ratón|  25.0|       2|  50.0|
|  3|2024-01-12| Teclado|  75.0|       1|  75.0|
|  4|2024-01-13| Monitor| 220.0|       1| 220.0|
|  5|2024-01-14|Portátil|1150.0|       2|2300.0|
+---+----------+--------+------+--------+------+
only showing top 5 rows


In [36]:
# pasamos la fecha de formato string -> fecha
df = df.withColumn("date", F.to_date(F.col("date"), "yyyy-mm-dd"))
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total: double (nullable = true)



In [37]:
#sacar el dia de la semana
df = df.withColumn("weekday", F.dayofweek(F.col("date")))
df.show(5)

+---+----------+--------+------+--------+------+-------+
| id|      date| product| price|quantity| total|weekday|
+---+----------+--------+------+--------+------+-------+
|  1|2024-01-10|Portátil|1200.0|       1|1200.0|      4|
|  2|2024-01-11|   Ratón|  25.0|       2|  50.0|      5|
|  3|2024-01-12| Teclado|  75.0|       1|  75.0|      6|
|  4|2024-01-13| Monitor| 220.0|       1| 220.0|      7|
|  5|2024-01-14|Portátil|1150.0|       2|2300.0|      1|
+---+----------+--------+------+--------+------+-------+
only showing top 5 rows


In [38]:
# agrupamos por dia de la semana para contar la cantidad
df_resumen = df.groupBy("weekday").agg(
    F.count("weekday").alias("cantidas de veces")
)
df_resumen.orderBy(F.col("weekday").desc()).show()

+-------+-----------------+
|weekday|cantidas de veces|
+-------+-----------------+
|      7|                3|
|      6|                3|
|      5|                3|
|      4|                3|
|      3|                2|
|      2|                3|
|      1|                3|
+-------+-----------------+



### Ejercicio 3 — Manejar registros corruptos / malformados
Imagina que alguien añadió una fila mal formada a sales_simple.csv (por ejemplo, 11,2024-01-20,Portátil,not_a_number,2 donde price NO es convertible a double). Crea un CSV sales_malformed.csv con esa fila extra. Lee el CSV de dos maneras:

A) usando option("mode", "DROPMALFORMED") y un schema explícito — comprueba el count final.

B) usando option("mode", "PERMISSIVE") y option("columnNameOfCorruptRecord", "_corrupt_record") con mismo schema — comprueba cómo Spark conserva el registro corrupto en una columna.

Piensa la diferencia y cuándo usar cada modo.

Objetivo
Saber manejar líneas malformadas sin romper el pipeline.

In [39]:
#definimos esquema
schema_mal = StructType([
    StructField("id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
])

In [40]:
# Tratamiento -> DROPMALFORMED: Elimina los registros corruptos
df_dropmalformed = spark.read.csv("sales_malformed.csv", header=True, schema=schema_mal, sep=",", mode="DROPMALFORMED")

In [41]:
df_dropmalformed.show()

+---+----------+-----------+------+--------+
| id|      date|    product| price|quantity|
+---+----------+-----------+------+--------+
|  1|2024-01-10|   Portátil|1200.0|       1|
|  2|2024-01-11|      Ratón|  25.0|       2|
|  3|2024-01-12|    Teclado|  75.0|       1|
|  4|2024-01-13|    Monitor| 220.0|       1|
|  5|2024-01-14|   Portátil|1150.0|       2|
|  6|2024-01-15|      Ratón|  20.0|       3|
|  7|2024-01-16|Auriculares|  60.0|       2|
|  8|2024-01-17|    Teclado|  80.0|       1|
|  9|2024-01-18|    Monitor| 210.0|       2|
| 10|2024-01-19|   Portátil|1250.0|       1|
| 11|2024-01-20|      Ratón|  22.0|       4|
| 12|2024-01-21|Auriculares|  55.0|       1|
| 13|2024-01-22|    Teclado|  70.0|       2|
| 14|2024-01-23|    Monitor| 230.0|       1|
| 15|2024-01-24|   Portátil|1190.0|       1|
| 16|2024-01-25|      Ratón|  24.0|       2|
| 17|2024-01-26|Auriculares|  65.0|       1|
| 18|2024-01-27|    Teclado|  85.0|       1|
| 19|2024-01-28|    Monitor| 215.0|       1|
| 20|2024-

In [42]:
df_dropmalformed.count()

21

In [43]:
# Tratamiento -> Permisivo: creamos una nueva columna que registra el fallo
# 1. modificamos esquema
#schema_mal = StructType(schema_mal.fields + [StructField("_corrupt_record", StringType(), True)])

# 2. lectura de valores
df_permisive = spark.read.csv("sales_malformed.csv", header=True, sep=",", schema=schema_mal,
                              mode="PERMISSIVE", columnNameOfCorruptRecord="_corrupt_record")

In [44]:
df_permisive.count()

21

In [45]:
df_permisive.show()

+---+----------+-----------+------+--------+
| id|      date|    product| price|quantity|
+---+----------+-----------+------+--------+
|  1|2024-01-10|   Portátil|1200.0|       1|
|  2|2024-01-11|      Ratón|  25.0|       2|
|  3|2024-01-12|    Teclado|  75.0|       1|
|  4|2024-01-13|    Monitor| 220.0|       1|
|  5|2024-01-14|   Portátil|1150.0|       2|
|  6|2024-01-15|      Ratón|  20.0|       3|
|  7|2024-01-16|Auriculares|  60.0|       2|
|  8|2024-01-17|    Teclado|  80.0|       1|
|  9|2024-01-18|    Monitor| 210.0|       2|
| 10|2024-01-19|   Portátil|1250.0|       1|
| 11|2024-01-20|      Ratón|  22.0|       4|
| 12|2024-01-21|Auriculares|  55.0|       1|
| 13|2024-01-22|    Teclado|  70.0|       2|
| 14|2024-01-23|    Monitor| 230.0|       1|
| 15|2024-01-24|   Portátil|1190.0|       1|
| 16|2024-01-25|      Ratón|  24.0|       2|
| 17|2024-01-26|Auriculares|  65.0|       1|
| 18|2024-01-27|    Teclado|  85.0|       1|
| 19|2024-01-28|    Monitor| 215.0|       1|
| 20|2024-

In [46]:
spark.stop()